In [1]:
!pip install -q langchain langchain-groq groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 1.6 MB/s eta 0:00:00


In [2]:
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate
from google.colab import userdata
import os

os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

print("✅ Setup complete")
print(f"🤖 Model: llama-3.3-70b-versatile")

✅ Setup complete
🤖 Model: llama-3.3-70b-versatile


In [3]:
@tool
def check_customer_risk(balance: float, transactions: int, is_active: bool) -> str:
    """Check the risk level of a banking customer based on their profile.
    Use this when asked about customer risk assessment."""

    if not is_active:
        return "HIGH RISK — Account inactive"
    if balance < 1000:
        return f"HIGH RISK — Low balance: €{balance}"
    elif balance < 5000:
        return f"MEDIUM RISK — Moderate balance: €{balance}"
    else:
        return f"LOW RISK — Healthy balance: €{balance}"

print("✅ Tool 1 created: check_customer_risk")
print(f"   Name: {check_customer_risk.name}")
print(f"   Description: {check_customer_risk.description}")

✅ Tool 1 created: check_customer_risk
   Name: check_customer_risk
   Description: Check the risk level of a banking customer based on their profile.
    Use this when asked about customer risk assessment.


In [4]:
@tool
def check_aml_flag(transaction_amount: float, country: str) -> str:
    """Check if a transaction requires AML screening or has compliance flags.
    Use this when asked about AML, suspicious transactions or country risk."""

    high_risk_countries = ["Test1", "Test2", "Test3", "Test4"]
    flags = []

    if transaction_amount > 10000:
        flags.append("Transaction above EUR 10,000 — monitoring required")
    if transaction_amount > 5000:
        flags.append("Cash transaction above EUR 5,000 — documentation required")
    if country in high_risk_countries:
        flags.append(f"{country} is high risk jurisdiction — Enhanced Due Diligence required")

    if flags:
        return "🚨 AML FLAGS:\n" + "\n".join(flags)
    return "✅ No AML flags — transaction within normal parameters"


@tool
def check_credit_limit(client_exposure: float, total_portfolio: float, sector: str) -> str:
    """Check if a client exposure breaches credit risk concentration limits.
    Use this when asked about credit limits, exposure or portfolio concentration."""

    exposure_pct = (client_exposure / total_portfolio) * 100
    issues = []

    if exposure_pct > 10:
        issues.append(f"⚠️ Single client exposure {exposure_pct:.1f}% exceeds 10% limit")

    sector_limit = 30 if sector.lower() == "real estate" else 25
    if exposure_pct > sector_limit:
        issues.append(f"⚠️ Sector concentration exceeds {sector_limit}% limit for {sector}")

    if client_exposure > 5000000:
        issues.append("⚠️ Exposure above EUR 5M — immediate escalation required")

    if issues:
        return "LIMIT BREACH:\n" + "\n".join(issues)
    return f"✅ Exposure {exposure_pct:.1f}% — within all limits"


# All tools together
tools = [check_customer_risk, check_aml_flag, check_credit_limit]

print(f"✅ All {len(tools)} tools ready:")
for t in tools:
    print(f"  → {t.name}")

✅ All 3 tools ready:
  → check_customer_risk
  → check_aml_flag
  → check_credit_limit


In [5]:
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage

system_message = """You are a senior banking compliance officer
at Northern Europe Bank.

You have access to these tools:
- check_customer_risk: assess customer risk level
- check_aml_flag: check AML compliance flags
- check_credit_limit: check credit exposure limits

Always use the appropriate tools to answer questions.
Think step by step and cite specific thresholds.
After using tools summarise your findings clearly."""

# Bind tools directly to the LLM
llm_with_tools = llm.bind_tools(tools)

print("✅ Agent ready")
print("🔧 Tools bound to LLM")
print("The LLM can now decide when to call each tool")

✅ Agent ready
🔧 Tools bound to LLM
The LLM can now decide when to call each tool


In [6]:
def run_agent(question):
    print(f"\n❓ Question: {question}")
    print("-" * 50)

    messages = [
        SystemMessage(content=system_message),
        HumanMessage(content=question)
    ]

    # Step 1 — LLM decides what to do
    response = llm_with_tools.invoke(messages)
    print(f"🤖 Agent thinking...")

    # Step 2 — Check if agent wants to use a tool
    if response.tool_calls:
        print(f"🔧 Agent decided to use tools:")

        for tool_call in response.tool_calls:
            tool_name = tool_call['name']
            tool_args = tool_call['args']
            print(f"   → Calling: {tool_name}")
            print(f"   → With args: {tool_args}")

            # Step 3 — Find and run the right tool
            for t in tools:
                if t.name == tool_name:
                    tool_result = t.invoke(tool_args)
                    print(f"   → Result: {tool_result}")

            # Step 4 — Add tool result to messages
            messages.append(response)
            from langchain_core.messages import ToolMessage
            messages.append(
                ToolMessage(
                    content=str(tool_result),
                    tool_call_id=tool_call['id']
                )
            )

        # Step 5 — LLM gives final answer using tool results
        final_response = llm_with_tools.invoke(messages)
        print(f"\n✅ FINAL ANSWER: {final_response.content}")

    else:
        # No tool needed — direct answer
        print(f"\n✅ ANSWER: {response.content}")

print("✅ Agent loop ready")

✅ Agent loop ready


In [7]:
# Test 1 — Customer Risk
run_agent("Check risk for customer with balance €750, 142 transactions, active account")


❓ Question: Check risk for customer with balance €750, 142 transactions, active account
--------------------------------------------------
🤖 Agent thinking...
🔧 Agent decided to use tools:
   → Calling: check_customer_risk
   → With args: {'balance': 750, 'is_active': True, 'transactions': 142}
   → Result: HIGH RISK — Low balance: €750.0

✅ FINAL ANSWER: The customer with a balance of €750, 142 transactions, and an active account has been assessed as HIGH RISK due to a low balance of €750.0.


In [8]:
# Test 2 — AML Check
run_agent("Is a €15,000 transaction from Test1 flagged for AML?")


❓ Question: Is a €15,000 transaction from Test1 flagged for AML?
--------------------------------------------------
🤖 Agent thinking...
🔧 Agent decided to use tools:
   → Calling: check_aml_flag
   → With args: {'country': 'Test1', 'transaction_amount': 15000}
   → Result: 🚨 AML FLAGS:
Transaction above EUR 10,000 — monitoring required
Cash transaction above EUR 5,000 — documentation required
Test1 is high risk jurisdiction — Enhanced Due Diligence required

✅ FINAL ANSWER: The €15,000 transaction from Test1 is flagged for AML screening due to the high-risk jurisdiction and the transaction amount exceeding the EUR 10,000 threshold. Enhanced Due Diligence is required for this transaction.


In [9]:
run_agent("Client exposure is €8M out of €50M portfolio in real estate sector — any breaches?")


❓ Question: Client exposure is €8M out of €50M portfolio in real estate sector — any breaches?
--------------------------------------------------
🤖 Agent thinking...
🔧 Agent decided to use tools:
   → Calling: check_credit_limit
   → With args: {'client_exposure': 8000000, 'sector': 'real estate', 'total_portfolio': 50000000}
   → Result: LIMIT BREACH:
⚠️ Single client exposure 16.0% exceeds 10% limit
⚠️ Exposure above EUR 5M — immediate escalation required

✅ FINAL ANSWER: Based on the check_credit_limit function, the client exposure of €8M out of a €50M portfolio in the real estate sector exceeds the 10% limit, with a 16.0% exposure. This breach requires immediate escalation.


In [10]:
run_agent("""
New client onboarding review:
- Account balance: €3,500
- Transactions: 89
- Account active: Yes
- Recent transaction: €12,000 from Test2
- Sector exposure: €6M out of €40M portfolio in manufacturing

Please do a complete compliance check.
""")


❓ Question: 
New client onboarding review:
- Account balance: €3,500
- Transactions: 89
- Account active: Yes
- Recent transaction: €12,000 from Test2
- Sector exposure: €6M out of €40M portfolio in manufacturing

Please do a complete compliance check.

--------------------------------------------------
🤖 Agent thinking...
🔧 Agent decided to use tools:
   → Calling: check_customer_risk
   → With args: {'balance': 3500, 'is_active': True, 'transactions': 89}
   → Result: MEDIUM RISK — Moderate balance: €3500.0
   → Calling: check_aml_flag
   → With args: {'country': 'Unknown', 'transaction_amount': 12000}
   → Result: 🚨 AML FLAGS:
Transaction above EUR 10,000 — monitoring required
Cash transaction above EUR 5,000 — documentation required
   → Calling: check_credit_limit
   → With args: {'client_exposure': 6000000, 'sector': 'manufacturing', 'total_portfolio': 40000000}
   → Result: LIMIT BREACH:
⚠️ Single client exposure 15.0% exceeds 10% limit
⚠️ Exposure above EUR 5M — immediate esca

In [11]:
print("🏦 Banking Compliance Agent")
print("Ask about customer risk, AML or credit limits")
print("Type 'exit' to quit\n")

while True:
    question = input("❓ Your question: ")
    if question.lower() == 'exit':
        print("👋 Goodbye!")
        break
    run_agent(question)

🏦 Banking Compliance Agent
Ask about customer risk, AML or credit limits
Type 'exit' to quit

❓ Your question: exit
👋 Goodbye!


In [12]:
!pip install -q langchain langchain-groq langchain-text-splitters langchain-huggingface langchain-community pypdf sentence-transformers faiss-cpu groq reportlab

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 53.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 74.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 50.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [13]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.messages import ToolMessage

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=50)

print("✅ RAG components added")

/tmp/ipykernel_1357/1139347148.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ RAG components added


In [14]:
from reportlab.lib.pagesizes import A4
from reportlab.pdfgen import canvas

def create_pdf(filename, title, content):
    c = canvas.Canvas(filename, pagesize=A4)
    width, height = A4
    c.setFont("Helvetica-Bold", 14)
    c.drawString(50, height-50, title)
    c.setFont("Helvetica", 9)
    y = height - 80
    for line in content:
        if y < 50:
            c.showPage()
            y = height - 50
        c.drawString(50, y, line)
        y -= 14
    c.save()
    print(f"✅ Created: {filename}")

create_pdf("esg_policy.pdf", "NORTHERN EUROPE BANK — ESG POLICY 2024", [
    "Section 1 — Scope",
    "Applies to corporate clients with turnover exceeding EUR 10 million.",
    "Section 2 — Reporting Requirements",
    "- GHG Scope 1 and 2 emissions in tonnes CO2",
    "- Energy consumption in MWh",
    "- Water usage in cubic metres",
    "Section 3 — EU Taxonomy",
    "DNSH assessment mandatory for Energy, Manufacturing and Transport.",
    "Section 4 — Deadlines",
    "Annual ESG reports due 31 March each year.",
    "Section 5 — Non-Compliance",
    "- Credit facility review",
    "- Potential loan covenant breach",
    "- Escalation to senior management",
])

create_pdf("credit_risk_policy.pdf", "NORTHERN EUROPE BANK — CREDIT RISK POLICY 2024", [
    "Section 1 — Risk Classification",
    "Green: score above 750 — standard terms",
    "Amber: score 650-750 — enhanced monitoring",
    "Red: score below 650 — credit committee approval",
    "Section 2 — Exposure Limits",
    "Single client exposure must not exceed 10% of total portfolio.",
    "Sector concentration limit is 25% of total portfolio.",
    "Real estate sector limit is 30%.",
    "Section 3 — Review Frequency",
    "Green clients: Annual review",
    "Amber clients: Semi-annual review",
    "Red clients: Quarterly review",
    "Section 4 — Reporting",
    "Credit risk reports due to Risk Committee monthly.",
    "Exceptions above EUR 5 million require immediate escalation.",
])

create_pdf("aml_policy.pdf", "NORTHERN EUROPE BANK — AML POLICY 2024", [
    "Section 1 — Customer Due Diligence",
    "Enhanced Due Diligence required for:",
    "- Politically Exposed Persons",
    "- High risk jurisdictions",
    "- Clients with turnover above EUR 50 million",
    "Section 2 — Transaction Monitoring",
    "Transactions above EUR 10,000 must be monitored.",
    "Suspicious transactions reported within 24 hours.",
    "Cash transactions above EUR 5,000 require documentation.",
    "Section 3 — Reporting",
    "Annual AML compliance report due 28 February.",
    "Sanctions screening performed daily.",
    "Section 4 — Penalties",
    "Non-compliance fines up to EUR 10 million.",
    "Criminal prosecution of responsible officers.",
])

print("\n✅ All policy documents created!")

✅ Created: esg_policy.pdf
✅ Created: credit_risk_policy.pdf
✅ Created: aml_policy.pdf

✅ All policy documents created!


In [15]:
# Load all policy documents
pdf_files = ["esg_policy.pdf", "credit_risk_policy.pdf", "aml_policy.pdf"]
all_documents = []

for pdf in pdf_files:
    loader = PyPDFLoader(pdf)
    docs = loader.load()
    for doc in docs:
        doc.metadata["source"] = pdf
    all_documents.extend(docs)
    print(f"✅ Loaded: {pdf}")

# Create vector store
all_chunks = splitter.split_documents(all_documents)
vectorstore = FAISS.from_documents(all_chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print(f"\n✅ Vector store ready — {len(all_chunks)} chunks")

# RAG prompt
rag_prompt = ChatPromptTemplate.from_template("""
You are a banking policy expert.
Answer using ONLY the provided policy documents.
Always cite which document your answer comes from.

Context:
{context}

Question: {question}

Answer with citations:""")

def format_docs(docs):
    formatted = []
    for doc in docs:
        source = doc.metadata.get("source", "Unknown")
        formatted.append(f"[{source}]\n{doc.page_content}")
    return "\n\n".join(formatted)

# RAG chain
rag_chain = (
    {"context": retriever | format_docs,
     "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

# NOW wrap RAG as a tool the agent can use
@tool
def search_bank_policies(question: str) -> str:
    """Search Northern Europe Bank internal policy documents.
    Use this when asked about bank policies, rules, deadlines,
    reporting requirements or compliance procedures."""
    return rag_chain.invoke(question)

print("✅ RAG wrapped as agent tool")

✅ Loaded: esg_policy.pdf
✅ Loaded: credit_risk_policy.pdf
✅ Loaded: aml_policy.pdf

✅ Vector store ready — 13 chunks
✅ RAG wrapped as agent tool


In [16]:
# All 4 tools together
all_tools = [
    check_customer_risk,
    check_aml_flag,
    check_credit_limit,
    search_bank_policies  # ← new RAG tool
]

system_message = """You are a senior banking compliance officer
at Northern Europe Bank.

You have access to these tools:
- check_customer_risk: assess customer risk level
- check_aml_flag: check AML compliance flags
- check_credit_limit: check credit exposure limits
- search_bank_policies: search internal bank policy documents

When answering:
1. Use calculation tools for numbers and limits
2. Use search_bank_policies for rules and procedures
3. Combine both for complete answers
4. Always cite sources"""

# Bind all 4 tools to LLM
llm_with_all_tools = llm.bind_tools(all_tools)

print(f"✅ Agent updated with {len(all_tools)} tools:")
for t in all_tools:
    print(f"  → {t.name}")

✅ Agent updated with 4 tools:
  → check_customer_risk
  → check_aml_flag
  → check_credit_limit
  → search_bank_policies


In [17]:
def run_agent(question):
    print(f"\n❓ Question: {question}")
    print("-" * 55)

    messages = [
        SystemMessage(content=system_message),
        HumanMessage(content=question)
    ]

    # Agent loop — max 5 iterations
    for i in range(5):
        response = llm_with_all_tools.invoke(messages)

        # No more tool calls — final answer
        if not response.tool_calls:
            print(f"\n✅ FINAL ANSWER:\n{response.content}")
            break

        # Agent wants to use tools
        print(f"\n🔧 Step {i+1} — Agent using tools:")
        messages.append(response)

        for tool_call in response.tool_calls:
            tool_name = tool_call['name']
            tool_args = tool_call['args']
            print(f"  → {tool_name}")
            print(f"     Args: {tool_args}")

            # Find and run the right tool
            tool_result = None
            for t in all_tools:
                if t.name == tool_name:
                    tool_result = t.invoke(tool_args)
                    break

            print(f"     Result: {str(tool_result)[:150]}...")

            # Add result back to messages
            messages.append(
                ToolMessage(
                    content=str(tool_result),
                    tool_call_id=tool_call['id']
                )
            )

print("✅ Updated agent ready")

✅ Updated agent ready


In [18]:
# Test 1 — Needs RAG only
print("=" * 55)
print("TEST 1 — Policy question")
run_agent("What are all the reporting deadlines in our bank?")

TEST 1 — Policy question

❓ Question: What are all the reporting deadlines in our bank?
-------------------------------------------------------

🔧 Step 1 — Agent using tools:
  → search_bank_policies
     Args: {'question': 'reporting deadlines Northern Europe Bank internal policy documents'}
     Result: According to the Northern Europe Bank's internal policy documents, the reporting deadlines are as follows: 
Suspicious transactions must be reported w...

🔧 Step 2 — Agent using tools:
  → search_bank_policies
     Args: {'question': 'reporting deadlines for ESG and AML reports Northern Europe Bank internal policy documents'}
     Result: According to the Northern Europe Bank's internal policy documents, the reporting deadlines are as follows:

For ESG reports, the deadline is 31 March ...

🔧 Step 3 — Agent using tools:
  → search_bank_policies
     Args: {'question': 'Northern Europe Bank reporting requirements and deadlines'}
     Result: According to the provided policy documents, 

In [19]:
system_message = """You are a senior banking compliance officer
at Northern Europe Bank.

You have access to these tools:
- check_customer_risk: assess customer risk level
- check_aml_flag: check AML compliance flags
- check_credit_limit: check credit exposure limits
- search_bank_policies: search internal bank policy documents

IMPORTANT RULES:
1. Call search_bank_policies MAXIMUM 1 time per question
2. Call calculation tools MAXIMUM 1 time each
3. After getting tool results — STOP and give final answer
4. Do NOT search multiple times for the same information
5. Synthesise all results into one clear answer"""

# Rebind tools with updated system message
llm_with_all_tools = llm.bind_tools(all_tools)

print("✅ System prompt updated — agent will be more decisive")

✅ System prompt updated — agent will be more decisive


In [20]:
print("=" * 55)
print("TEST 1 — Policy question")
run_agent("What are all the reporting deadlines in our bank?")

TEST 1 — Policy question

❓ Question: What are all the reporting deadlines in our bank?
-------------------------------------------------------

🔧 Step 1 — Agent using tools:
  → search_bank_policies
     Args: {'question': 'reporting deadlines Northern Europe Bank internal policy documents'}
     Result: According to the Northern Europe Bank's internal policy documents, the reporting deadlines are as follows: 
Suspicious transactions must be reported w...

✅ FINAL ANSWER:
According to the Northern Europe Bank's internal policy documents, the reporting deadlines are as follows: 
Suspicious transactions must be reported within 24 hours (aml_policy.pdf, Section 3 — Reporting). 
There are no other reporting deadlines specified in the provided documents (esg_policy.pdf, Section 1 — Scope and Section 2 — Reporting Requirements; aml_policy.pdf, Section 1 — Customer Due Diligence and Section 3 — Reporting).


In [21]:
print("=" * 55)
print("TEST 2 — Numbers + Policy combined")
run_agent("""
Complete onboarding check for new client:
- Balance: €3,500, 89 transactions, active
- Transaction: €12,000 from Test2
- Exposure: €6M out of €40M in manufacturing
- Also tell me what our policy says about
  enhanced due diligence requirements
""")

TEST 2 — Numbers + Policy combined

❓ Question: 
Complete onboarding check for new client:
- Balance: €3,500, 89 transactions, active
- Transaction: €12,000 from Test2
- Exposure: €6M out of €40M in manufacturing
- Also tell me what our policy says about 
  enhanced due diligence requirements

-------------------------------------------------------

🔧 Step 1 — Agent using tools:
  → check_customer_risk
     Args: {'balance': 3500, 'is_active': True, 'transactions': 89}
     Result: MEDIUM RISK — Moderate balance: €3500.0...
  → check_aml_flag
     Args: {'country': 'Test2', 'transaction_amount': 12000}
     Result: 🚨 AML FLAGS:
Transaction above EUR 10,000 — monitoring required
Cash transaction above EUR 5,000 — documentation required
Test2 is high risk jurisdict...
  → check_credit_limit
     Args: {'client_exposure': 6000000, 'sector': 'manufacturing', 'total_portfolio': 40000000}
     Result: LIMIT BREACH:
⚠️ Single client exposure 15.0% exceeds 10% limit
⚠️ Exposure above EUR 5M — 